In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('sfbench.csv')
df['MachineSeries'] = df['MachineType'].apply(lambda x: x.split('-')[0])
df.info()

In [ ]:
mean_nps = df.groupby(['StockfishBinary', 'StockfishVersion', 'MachineType', 'MachineSeries', 'Threads', 'CpuProcessors'])['NPS'].mean().reset_index()
def config_summary(row):
    bsummary = row['StockfishBinary'].rsplit('-', maxsplit=1)[-1]
    return f'{bsummary} {row["MachineType"]}'
mean_nps['ConfigSummary'] = mean_nps.apply(config_summary, axis=1)
mean_nps['NPSPerThread'] = mean_nps.apply(lambda r: r['NPS'] / r['Threads'], axis=1)
mean_nps

In [ ]:
import colorsys
import plotly.colors as pc

# D3's qualitative palette is designed for perceptual distinctiveness between categories
base_palette = pc.qualitative.D3

series_list = sorted(mean_nps['MachineSeries'].unique())
configs_per_series = {
    s: sorted(mean_nps[mean_nps['MachineSeries'] == s]['ConfigSummary'].unique())
    for s in series_list
}

def hex_to_hls(hex_color):
    hex_color = hex_color.lstrip('#')
    r, g, b = [int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4)]
    return colorsys.rgb_to_hls(r, g, b)

color_map = {}
for i, series in enumerate(series_list):
    h, l, s = hex_to_hls(base_palette[i % len(base_palette)])
    configs = configs_per_series[series]
    n_configs = len(configs)
    for j, config in enumerate(configs):
        lightness = 0.35 + 0.3 * (j / max(n_configs - 1, 1))
        r, g, b = colorsys.hls_to_rgb(h, lightness, s)
        color_map[config] = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'


In [ ]:
import plotly.graph_objects as go

def add_cpu_limit_markers(fig, df, y_col, color_map):
    """Add a filled dot on each line where Threads == CpuProcessors."""
    marker_df = df[df['Threads'] == df['CpuProcessors']]
    for _, row in marker_df.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Threads']],
            y=[row[y_col]],
            mode='markers',
            marker=dict(size=7, color=color_map.get(row['ConfigSummary'], 'black'), symbol='circle'),
            name=row['ConfigSummary'],
            showlegend=False,
        ))

In [ ]:
s18_df = mean_nps[mean_nps['StockfishVersion'] == 18.0]
fig = px.line(s18_df, x='Threads', y='NPS', color='ConfigSummary', title='Stockfish 18 benchmark (Ubuntu x86)',
              width=800, height=600, color_discrete_map=color_map)
fig.update_layout(yaxis_title='Mean NPS')
add_cpu_limit_markers(fig, s18_df, 'NPS', color_map)
fig.show()

In [ ]:
fig = px.line(s18_df, x='Threads', y='NPSPerThread', color='ConfigSummary', title='Stockfish 18 benchmark (Ubuntu x86)',
              width=800, height=600, color_discrete_map=color_map)
fig.update_layout(yaxis_title='Mean NPS per Thread')
add_cpu_limit_markers(fig, s18_df, 'NPSPerThread', color_map)
fig.show()

In [ ]:
s161_df = mean_nps[mean_nps['StockfishVersion'] == 16.1]
fig = px.line(s161_df, x='Threads', y='NPS', color='ConfigSummary',
             color_discrete_map=color_map,
             title='Stockfish 16.1 benchmark (Ubuntu x86)', width=800, height=600)
fig.update_layout(yaxis_title='Mean NPS')
add_cpu_limit_markers(fig, s161_df, 'NPS', color_map)
fig.show()

In [ ]:
fig = px.line(s161_df[s161_df['Threads'] <= s161_df['CpuProcessors']], x='Threads', y='NPSPerThread', color='ConfigSummary',
             color_discrete_map=color_map,
             title='Stockfish 16.1 benchmark (Ubuntu x86)', width=800, height=600)
fig.update_layout(yaxis_title='Mean NPS per thread')
add_cpu_limit_markers(fig, s161_df, 'NPSPerThread', color_map)
fig.show()